In [38]:
import pyarrow.parquet as pq
import os 
import duckdb
import time

# User of Interests - Scraped Data
Users who have blocked someone in their first week.

**Data Source:** Firehose API crawl from Dec 31 to Jan 16, 2026

**Note:** This is relative sampling - we pick the join date as the reference point.

In [39]:
# Load the blocks database and profiles using DuckDB (memory efficient for large files)
blocks_path = "../../data/scraped/cleaned/blocks.parquet"
profiles_path = "../../data/scraped/cleaned/profiles.parquet"

# Connect to DuckDB
conn = duckdb.connect()

print("Analyzing blocks in first week after joining using DuckDB...")

# Get basic info about the data
total_blocks = conn.execute(f"SELECT COUNT(*) FROM read_parquet('{blocks_path}')").fetchone()[0]
total_profiles = conn.execute(f"SELECT COUNT(*) FROM read_parquet('{profiles_path}')").fetchone()[0]

print(f"Total blocks: {total_blocks:,}")
print(f"Total profiles: {total_profiles:,}")

Analyzing blocks in first week after joining using DuckDB...
Total blocks: 142,307
Total profiles: 81,798


In [40]:
parquet_file = pq.ParquetFile(blocks_path)
metadata = parquet_file.metadata
schema = parquet_file.schema_arrow

print(schema)

did_id: int64
created_date: string
subject_did_id: int64
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 622


In [41]:
# Query to find blocks in the first week after joining (days 0-6)
# Restrict to users who joined between Jan 1-3, 2026 (to ensure 14+ days of data)
first_week_blocks_query = f"""
WITH blocks_with_join_days AS (
    SELECT 
        b.did_id,
        b.created_date AS block_date,
        p.created_date AS join_date,
        DATE_DIFF('day', CAST(p.created_date AS TIMESTAMP), CAST(b.created_date AS TIMESTAMP)) AS days_since_join
    FROM read_parquet('{blocks_path}') b
    INNER JOIN read_parquet('{profiles_path}') p
        ON b.did_id = p.did_id
    WHERE DATE_DIFF('day', CAST(p.created_date AS TIMESTAMP), CAST(b.created_date AS TIMESTAMP)) BETWEEN 0 AND 6
)
SELECT
    did_id,
    join_date,
    COUNT(*) AS block_count
FROM blocks_with_join_days
GROUP BY did_id, join_date
"""

# Execute query and get results
first_week_blocks = conn.execute(first_week_blocks_query).df()

print(f"\nUnique users who blocked someone in first week: {len(first_week_blocks):,}")
print(f"Total blocks in first week: {first_week_blocks['block_count'].sum():,}")



Unique users who blocked someone in first week: 1,081
Total blocks in first week: 4,061


In [42]:
# Save ALL users who blocked someone in their first week (no sampling for test data)
first_week_blockers_path = "../../data/ale_simplicistic_model/scraped/filtered/first_week_blockers.parquet"

first_week_blockers = (
    first_week_blocks[['did_id', 'join_date']]
    .drop_duplicates()
    .reset_index(drop=True)
)

first_week_blockers.to_parquet(first_week_blockers_path, index=False)

print(f"Saved {len(first_week_blockers):,} first-week blocker user ids to: {first_week_blockers_path}")

Saved 1,081 first-week blocker user ids to: ../../data/ale_simplicistic_model/scraped/filtered/first_week_blockers.parquet


# Event-Activity Filtering
The criteria for filtering are:
- Events must involve at least one user of interest.
- Events must occur within one week after the user's join date.

In [43]:
def filter_events(input_path, output_path, db_name, query):

    print(f"Filtering {db_name} using DuckDB...")
    start_time = time.time()
    
    conn = duckdb.connect()
    
    # Register join_date
    users_of_interests = conn.from_df(first_week_blockers)
    events_table = conn.read_parquet(input_path)
    
    # Get total row count before filtering
    total_rows = conn.execute("SELECT COUNT(*) FROM events_table").fetchall()[0][0]
    
    # Execute the provided query
    filtered_events = conn.execute(query).fetch_arrow_table()
    filtered_rows = filtered_events.num_rows
    
    # Create directory if it doesn't exist
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    # Write to parquet
    pq.write_table(filtered_events, output_path, compression='zstd')
    
    elapsed_time = time.time() - start_time
    input_size = os.path.getsize(input_path)
    output_size = os.path.getsize(output_path)
    retention_rate = (filtered_rows / total_rows * 100) if total_rows > 0 else 0
    
    print(f"\n✅ DuckDB filtering completed in {elapsed_time:.2f} seconds")
    print(f"Rows: {total_rows:,} → {filtered_rows:,} ({retention_rate:.1f}% kept)")
    print(f"File size: {input_size / (1024**3):.2f} GB → {output_size / (1024**3):.2f} GB")
    print(f"Output: {output_path}")
    
    conn.close()
    
    return {
        'elapsed_time': elapsed_time,
        'total_rows': total_rows,
        'filtered_rows': filtered_rows,
        'retention_rate': retention_rate,
        'input_size_gb': input_size / (1024**3),
        'output_size_gb': output_size / (1024**3)
    }

In [44]:
# Filter blocks with 7-day window (two-user events) 
blocks_input_path = "../../data/scraped/cleaned/blocks.parquet"
blocks_output_path = "../../data/ale_simplicistic_model/scraped/filtered/blocks.parquet"

blocks_query = """
SELECT DISTINCT e.*
FROM events_table e
LEFT JOIN users_of_interests uoi_act ON e.did_id = uoi_act.did_id
LEFT JOIN users_of_interests uoi_sub ON e.subject_did_id = uoi_sub.did_id
WHERE 
    -- Keep if did_id is in users AND within 7 days of joining
    ((uoi_act.did_id IS NOT NULL 
      AND DATE_DIFF('day', CAST(uoi_act.join_date AS TIMESTAMP), CAST(e.created_date AS TIMESTAMP)) BETWEEN 0 AND 7)
    OR
    -- Keep if subject_did_id is in users AND within 7 days of joining
    (uoi_sub.did_id IS NOT NULL 
     AND DATE_DIFF('day', CAST(uoi_sub.join_date AS TIMESTAMP), CAST(e.created_date AS TIMESTAMP)) BETWEEN 0 AND 7))
"""

stats_blocks = filter_events(blocks_input_path, blocks_output_path, "blocks", blocks_query)
print(f"Blocks filtered: {stats_blocks['filtered_rows']:,} blocks from {stats_blocks['total_rows']:,} total")

Filtering blocks using DuckDB...

✅ DuckDB filtering completed in 0.05 seconds
Rows: 142,307 → 3,760 (2.6% kept)
File size: 0.00 GB → 0.00 GB
Output: ../../data/ale_simplicistic_model/scraped/filtered/blocks.parquet
Blocks filtered: 3,760 blocks from 142,307 total


In [45]:
# Filter posts with 7-day window (single-user events)
posts_input_path = "../../data/scraped/cleaned/posts.parquet"
posts_output_path = "../../data/ale_simplicistic_model/scraped/filtered/posts.parquet"

posts_query = """
SELECT e.*
FROM events_table e
LEFT JOIN users_of_interests uoi ON e.did_id = uoi.did_id
WHERE uoi.did_id IS NOT NULL
  AND DATE_DIFF('day', CAST(uoi.join_date AS TIMESTAMP), CAST(e.created_date AS TIMESTAMP)) BETWEEN 0 AND 7
"""

stats_posts = filter_events(posts_input_path, posts_output_path, "posts", posts_query)
print(f"Posts filtered: {stats_posts['filtered_rows']:,} posts from {stats_posts['total_rows']:,} total")

Filtering posts using DuckDB...

✅ DuckDB filtering completed in 0.15 seconds
Rows: 2,366,489 → 6,596 (0.3% kept)
File size: 0.02 GB → 0.00 GB
Output: ../../data/ale_simplicistic_model/scraped/filtered/posts.parquet
Posts filtered: 6,596 posts from 2,366,489 total


In [46]:
# Filter follows with 7-day window (two-user events)
follows_input_path = "../../data/scraped/cleaned/follows.parquet"
follows_output_path = "../../data/ale_simplicistic_model/scraped/filtered/follows.parquet"

follows_query = """
SELECT DISTINCT e.*
FROM events_table e
LEFT JOIN users_of_interests uoi_act ON e.did_id = uoi_act.did_id
LEFT JOIN users_of_interests uoi_sub ON e.subject_did_id = uoi_sub.did_id
WHERE 
    -- Keep if did_id is in users AND within 7 days of joining
    ((uoi_act.did_id IS NOT NULL 
      AND DATE_DIFF('day', CAST(uoi_act.join_date AS TIMESTAMP), CAST(e.created_date AS TIMESTAMP)) BETWEEN 0 AND 7)
    OR
    -- Keep if subject_did_id is in users AND within 7 days of joining
    (uoi_sub.did_id IS NOT NULL 
     AND DATE_DIFF('day', CAST(uoi_sub.join_date AS TIMESTAMP), CAST(e.created_date AS TIMESTAMP)) BETWEEN 0 AND 7))
"""

stats_follows = filter_events(follows_input_path, follows_output_path, "follows", follows_query)
print(f"Follows filtered: {stats_follows['filtered_rows']:,} follows from {stats_follows['total_rows']:,} total")

Filtering follows using DuckDB...

✅ DuckDB filtering completed in 0.08 seconds
Rows: 1,443,886 → 5,393 (0.4% kept)
File size: 0.01 GB → 0.00 GB
Output: ../../data/ale_simplicistic_model/scraped/filtered/follows.parquet
Follows filtered: 5,393 follows from 1,443,886 total


In [47]:
# Filter likes with 7-day window (two-user events)
likes_input_path = "../../data/scraped/cleaned/likes.parquet"
likes_output_path = "../../data/ale_simplicistic_model/scraped/filtered/likes.parquet"

likes_query = """
SELECT DISTINCT e.*
FROM events_table e
LEFT JOIN users_of_interests uoi_act ON e.did_id = uoi_act.did_id
LEFT JOIN users_of_interests uoi_sub ON e.subject_did_id = uoi_sub.did_id
WHERE 
    ((uoi_act.did_id IS NOT NULL 
      AND DATE_DIFF('day', CAST(uoi_act.join_date AS TIMESTAMP), CAST(e.created_date AS TIMESTAMP)) BETWEEN 0 AND 7)
    OR
    (uoi_sub.did_id IS NOT NULL 
     AND DATE_DIFF('day', CAST(uoi_sub.join_date AS TIMESTAMP), CAST(e.created_date AS TIMESTAMP)) BETWEEN 0 AND 7))
"""

stats_likes = filter_events(likes_input_path, likes_output_path, "likes", likes_query)
print(f"Likes filtered: {stats_likes['filtered_rows']:,} likes from {stats_likes['total_rows']:,} total")

Filtering likes using DuckDB...

✅ DuckDB filtering completed in 0.26 seconds
Rows: 12,650,613 → 14,820 (0.1% kept)
File size: 0.10 GB → 0.00 GB
Output: ../../data/ale_simplicistic_model/scraped/filtered/likes.parquet
Likes filtered: 14,820 likes from 12,650,613 total


In [48]:
# Summary
print("\n" + "="*80)
print("FILTERING SUMMARY")
print("="*80)
print(f"Users of interest: {len(first_week_blockers):,}")
print(f"\nFiltered events:")
print(f"  Blocks:  {stats_blocks['filtered_rows']:,} ({stats_blocks['retention_rate']:.1f}% of total)")
print(f"  Posts:   {stats_posts['filtered_rows']:,} ({stats_posts['retention_rate']:.1f}% of total)")
print(f"  Follows: {stats_follows['filtered_rows']:,} ({stats_follows['retention_rate']:.1f}% of total)")
print(f"  Likes:   {stats_likes['filtered_rows']:,} ({stats_likes['retention_rate']:.1f}% of total)")

conn.close()


FILTERING SUMMARY
Users of interest: 1,081

Filtered events:
  Blocks:  3,760 (2.6% of total)
  Posts:   6,596 (0.3% of total)
  Follows: 5,393 (0.4% of total)
  Likes:   14,820 (0.1% of total)
